In [1]:
# STEP 1 FIX — ROBUST DATA DOWNLOAD

import pandas as pd
import numpy as np
import yfinance as yf
import time

stocks = [
"RELIANCE.NS","HDFCBANK.NS","ICICIBANK.NS","SBIN.NS","AXISBANK.NS","KOTAKBANK.NS","INDUSINDBK.NS",
"TCS.NS","INFY.NS","HCLTECH.NS","WIPRO.NS","TECHM.NS",
"LT.NS","LTIM.NS","ULTRACEMCO.NS","GRASIM.NS","SHREECEM.NS",
"MARUTI.NS","M&M.NS","TATAMOTORS.NS","BAJAJ-AUTO.NS","EICHERMOT.NS",
"HINDUNILVR.NS","ITC.NS","NESTLEIND.NS","BRITANNIA.NS","DABUR.NS","GODREJCP.NS",
"BHARTIARTL.NS","ADANIENT.NS","ADANIPORTS.NS","POWERGRID.NS","NTPC.NS","TATAPOWER.NS",
"JSWSTEEL.NS","TATASTEEL.NS","HINDALCO.NS","COALINDIA.NS",
"ONGC.NS","BPCL.NS","IOC.NS","GAIL.NS",
"DRREDDY.NS","SUNPHARMA.NS","CIPLA.NS","DIVISLAB.NS","LUPIN.NS",
"BAJFINANCE.NS","BAJAJFINSV.NS","HDFCLIFE.NS","SBILIFE.NS","ICICIPRULI.NS",
"DLF.NS","LODHA.NS","GODREJPROP.NS",
"APOLLOHOSP.NS","FORTIS.NS",
"ZOMATO.NS","NYKAA.NS","PAYTM.NS"
]

benchmark = "^NSEI"

# 🔹 Batch download function
def download_in_batches(tickers, batch_size=10):
    all_data = []

    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        print(f"Downloading batch {i//batch_size + 1}...")

        try:
            data = yf.download(batch, start="2018-01-01")["Close"]
            all_data.append(data)
        except:
            print("Error in batch:", batch)

        time.sleep(1)  # avoid rate limits

    return pd.concat(all_data, axis=1)


# 🔹 Download stock data
stock_data = download_in_batches(stocks)

# 🔹 Benchmark separately
benchmark_data = yf.download(benchmark, start="2018-01-01")["Close"]

# 🔹 Align and clean data FIRST
stock_data = stock_data.sort_index()

# Forward fill small gaps
stock_data = stock_data.ffill()

# Drop columns that are mostly empty
stock_data = stock_data.dropna(axis=1, thresh=int(0.7 * len(stock_data)))

# Now compute returns
stock_returns = stock_data.pct_change(fill_method=None)

# Drop only initial NaNs
stock_returns = stock_returns.dropna(how="all")

print("Cleaned data shape:", stock_returns.shape)
nifty_returns = benchmark_data.pct_change(fill_method=None).dropna()

print("Final data shape:", stock_returns.shape)

[*********************100%***********************]  10 of 10 completed


[**********************70%*********              ]  7 of 10 completed$LTIM.NS: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-05-11) (Yahoo error = "No data found, symbol may be delisted")
[**********************80%*************          ]  8 of 10 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found
[*********************100%***********************]  10 of 10 completed

2 Failed downloads:
['LTIM.NS']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-05-11) (Yahoo error = "No data found, symbol may be delisted")
['TATAMOTORS.NS']: possibly delisted; no timezone found


[*********************100%***********************]  10 of 10 completed


[*********************100%***********************]  10 of 10 completed


[*********************100%***********************]  10 of 10 completed


[**********************80%*************          ]  8 of 10 completed$ZOMATO.NS: possibly delisted; no timezone found
[*********************100%***********************]  9 of 10 completed

1 Failed download:
['ZOMATO.NS']: possibly delisted; no timezone found
[*********************100%***********************]  1 of 1 completed

Cleaned data shape: (2062, 54)
Final data shape: (2062, 54)


In [2]:
#STEP-2 HIGH BETA STOCK FILTERED
import statsmodels.api as sm

# 🔹 Align dates between stocks and NIFTY
common_index = stock_returns.index.intersection(nifty_returns.index)

stock_returns_aligned = stock_returns.loc[common_index]
nifty_returns_aligned = nifty_returns.loc[common_index]

# 🔹 Beta function
def calc_beta(y, x):
    x = sm.add_constant(x)
    model = sm.OLS(y, x).fit()
    return model.params[1]

# 🔹 Compute beta for all stocks
betas = {}

for col in stock_returns_aligned.columns:
    try:
        betas[col] = calc_beta(stock_returns_aligned[col], nifty_returns_aligned)
    except:
        continue

betas = pd.Series(betas)

# 🔹 Filter HIGH BETA stocks (>1)
high_beta_stocks = betas[betas > 1].index

# 🔹 Filter returns
filtered_returns = stock_returns_aligned[high_beta_stocks]

print("Total stocks:", len(stock_returns_aligned.columns))
print("High beta stocks:", len(high_beta_stocks))
print("Sample betas:\n", betas.head())

Total stocks: 54
High beta stocks: 25
Sample betas:
 AXISBANK.NS      1.335667
HCLTECH.NS       0.764771
HDFCBANK.NS      1.052967
ICICIBANK.NS     1.262860
INDUSINDBK.NS    1.465673
dtype: float64


C:\Users\priya\AppData\Local\Temp\ipykernel_33752\3159287155.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return model.params[1]
C:\Users\priya\AppData\Local\Temp\ipykernel_33752\3159287155.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return model.params[1]
C:\Users\priya\AppData\Local\Temp\ipykernel_33752\3159287155.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return model.params[1]
C:\Users\priya\AppData\Lo

In [ ]:
def download_ohlc_batches(tickers, batch_size=10):
    all_data = []

    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        print(f"Downloading OHLC batch {i//batch_size + 1}...")

        try:
            data = yf.download(batch, start="2018-01-01")
            all_data.append(data)
        except:
            print("Error in OHLC batch:", batch)

        time.sleep(1)

    return pd.concat(all_data, axis=1)


# 🔹 Use this instead
ohlc = download_ohlc_batches(stocks)

In [ ]:
available_stocks = [s for s in high_beta_stocks if s in ohlc.columns.get_level_values(1)]

In [ ]:
def compute_adx(data, adx_threshold):
    data['adx'] = calculate_adx(data, period=14)['ADX']
    data['adx_signal'] = np.where(data.adx > adx_threshold, 1, 0)    
    data.iloc[:14,-1] = np.nan
    return data


def calculate_adx(df, period=14):
    # Calculate True Range (TR)
    df['H-L'] = df['High'] - df['Low']
    df['H-PC'] = np.abs(df['High'] - df['Close'].shift(1))
    df['L-PC'] = np.abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['H-L', 'H-PC', 'L-PC']].max(axis=1)

    # Calculate +DM and -DM
    df['+DM'] = np.where(df['High'] - df['High'].shift(1) > df['Low'].shift(1) - df['Low'], 
                         df['High'] - df['High'].shift(1), 0)
    df['-DM'] = np.where(df['Low'].shift(1) - df['Low'] > df['High'] - df['High'].shift(1), 
                         df['Low'].shift(1) - df['Low'], 0)

    # Smooth the TR, +DM, and -DM
    df['TR_smooth'] = df['TR'].rolling(window=period).sum()
    df['+DM_smooth'] = df['+DM'].rolling(window=period).sum()
    df['-DM_smooth'] = df['-DM'].rolling(window=period).sum()

    # Calculate +DI and -DI
    df['+DI'] = (df['+DM_smooth'] / df['TR_smooth']) * 100
    df['-DI'] = (df['-DM_smooth'] / df['TR_smooth']) * 100

    # Calculate ADX (the smoothed version of the absolute difference between +DI and -DI)
    df['ADX'] = np.nan
    df['ADX'] = df[['+DI', '-DI']].apply(lambda x: np.abs(x[0] - x[1]), axis=1)
    df['ADX'] = df['ADX'].rolling(window=period).mean()

    return df[['ADX', '+DI', '-DI']]

In [ ]:
# STEP 3 — SIGNALS

signals = pd.DataFrame(index=filtered_returns.index)

adx_threshold = 15  # you can tune later

for stock in high_beta_stocks:
    try:
        # Get OHLC for that stock
        df = ohlc.xs(stock, level=1, axis=1).copy()

        # Align with returns index
        df = df.loc[signals.index]

        # 🔹 Apply YOUR ADX
        df = compute_adx(df, adx_threshold)

        # 🔹 Momentum (simple)
        momentum = filtered_returns[stock].rolling(20).mean()

        # 🔥 FINAL SIGNAL
        signals[stock] = df['adx_signal'] * np.sign(momentum)

    except:
        continue

# Clean signals
signals = signals.dropna(how="all")

print("Signals shape:", signals.shape)
signals.tail()

In [ ]:
print(df['adx'].describe())

In [ ]:
# STEP 4 — COVARIANCE

# 🔹 Align signals & returns (IMPORTANT)
common_index = signals.index.intersection(filtered_returns.index)

signals_aligned = signals.loc[common_index]
returns_aligned = filtered_returns.loc[common_index]

# 🔹 Drop stocks where signals are all NaN
valid_stocks = signals_aligned.columns[signals_aligned.notna().sum() > 50]

signals_aligned = signals_aligned[valid_stocks]
returns_aligned = returns_aligned[valid_stocks]

# 🔹 Covariance matrix
cov_matrix = returns_aligned.cov()

# 🔹 Inverse covariance (use pseudo-inverse for stability)
inv_cov = np.linalg.pinv(cov_matrix.values)

print("Covariance shape:", cov_matrix.shape)

In [ ]:
# STEP 5 — RAW WEIGHTS

# 🔹 Take latest signals (most recent date)
latest_signal = signals_aligned.iloc[-1].values

# 🔹 Apply inverse covariance
raw_weights = inv_cov @ latest_signal

# 🔹 Convert to pandas Series
raw_weights = pd.Series(raw_weights, index=signals_aligned.columns)

print("Raw weights:")
print(raw_weights.sort_values(ascending=False).head())

In [ ]:
print(betas.loc[raw_weights.index].sort_values(ascending=False))

In [ ]:
print(high_beta_stocks)
print(raw_weights.index)

In [ ]:
# STEP 6 — RISK ADJUSTMENT

# 🔹 Volatility (std dev)
vol = returns_aligned.std()

# 🔹 Align with weights
vol = vol[raw_weights.index]

# 🔹 Risk-adjusted weights
weights_rp = raw_weights / vol

print("Risk-adjusted weights:")
print(weights_rp.sort_values(ascending=False).head())

In [ ]:
# STEP 7 — MARKET TREND (RP TREND)

# 🔹 Build simple Risk Parity weights (inverse vol)
rp_weights = 1 / vol
rp_weights = rp_weights / rp_weights.sum()

# 🔹 Create RP portfolio returns
rp_returns = (returns_aligned * rp_weights).sum(axis=1)

# 🔹 Trend of RP portfolio
rp_momentum = rp_returns.rolling(20).mean()

# 🔹 Market regime signal
rp_signal = np.sign(rp_momentum.iloc[-1])

print("RP Signal (Market Trend):", rp_signal)

In [ ]:
# STEP 8 — FINAL WEIGHTS

# 🔹 Apply market regime
final_weights = weights_rp * rp_signal

# 🔹 Normalize (important)
final_weights = final_weights / np.sum(np.abs(final_weights))

print("Final portfolio weights:")
print(final_weights.sort_values(ascending=False))